# 🎵 Music Playlist Generator
**Name:** Koina Chatterjee  
**Roll No:** AIML-A6/9951  
**Batch:** June 2026  
**Project Type:** Minor Project  
**Tech Stack:** Python, Pandas, Scikit-learn, Spotipy (Spotify API), Collaborative Filtering

## Step 1: Install & Import Libraries

In [ ]:
!pip install spotipy pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Connect to Spotify API

> **How to get free Spotify API credentials:**
> 1. Go to https://developer.spotify.com/dashboard
> 2. Log in with your Spotify account (free account works)
> 3. Click **Create App** → fill in name and description
> 4. Copy your **Client ID** and **Client Secret**
> 5. Paste them below

In [ ]:
# ===== PASTE YOUR SPOTIFY CREDENTIALS HERE =====
CLIENT_ID     = 'your_client_id_here'
CLIENT_SECRET = 'your_client_secret_here'
# ================================================

def connect_spotify(client_id, client_secret):
    """Connect to Spotify API using client credentials."""
    auth_manager = SpotifyClientCredentials(
        client_id=client_id,
        client_secret=client_secret
    )
    return spotipy.Spotify(auth_manager=auth_manager)

if CLIENT_ID != 'your_client_id_here':
    sp = connect_spotify(CLIENT_ID, CLIENT_SECRET)
    results = sp.search(q='Blinding Lights', limit=1)
    print('Connected to Spotify API!')
    print(f"Test track: {results['tracks']['items'][0]['name']}")
else:
    sp = None
    print('Using sample dataset (Spotify credentials not set).')

## Step 3: Load Song Dataset with Audio Features

In [ ]:
# Sample dataset with Spotify-style audio features
# In a live setup, these are fetched via sp.audio_features(track_id)
songs_data = {
    'track_name': [
        'Blinding Lights','Levitating','As It Was','Anti-Hero','Stay',
        'Good 4 U','Heat Waves','Watermelon Sugar','HUMBLE.','God Plan',
        'Sicko Mode','Old Town Road','Enemy','Believer','Mr. Brightside',
        'Animals','Wake Me Up','Faded','Take Five','Fly Me to the Moon',
        'Roses','Titanium','Rockstar','Thunder','Sunflower'
    ],
    'artist': [
        'The Weeknd','Dua Lipa','Harry Styles','Taylor Swift','The Kid LAROI',
        'Olivia Rodrigo','Glass Animals','Harry Styles','Kendrick Lamar','Drake',
        'Travis Scott','Lil Nas X','Imagine Dragons','Imagine Dragons','The Killers',
        'Martin Garrix','Avicii','Alan Walker','Dave Brubeck','Frank Sinatra',
        'SAINt JHN','David Guetta','Post Malone','Imagine Dragons','Post Malone'
    ],
    'genre': [
        'Pop','Pop','Pop','Pop','Pop',
        'Pop','Pop','Pop','Hip-Hop','Hip-Hop',
        'Hip-Hop','Hip-Hop','Rock','Rock','Rock',
        'Electronic','Electronic','Electronic','Jazz','Jazz',
        'Electronic','Electronic','Hip-Hop','Rock','Pop'
    ],
    'bpm': [
        171,103,174,97,170,
        167,80,95,150,77,
        155,136,137,125,148,
        128,124,90,172,110,
        99,126,160,168,100
    ],
    'energy': [
        0.73,0.76,0.73,0.64,0.85,
        0.90,0.51,0.82,0.62,0.45,
        0.73,0.65,0.83,0.84,0.93,
        0.97,0.61,0.65,0.31,0.29,
        0.59,0.78,0.53,0.59,0.55
    ],
    'valence': [
        0.89,0.91,0.83,0.58,0.79,
        0.71,0.73,0.95,0.43,0.35,
        0.50,0.59,0.48,0.46,0.49,
        0.63,0.55,0.27,0.65,0.82,
        0.48,0.44,0.19,0.55,0.91
    ],
    'danceability': [
        0.51,0.70,0.52,0.65,0.59,
        0.56,0.66,0.55,0.91,0.75,
        0.74,0.76,0.56,0.49,0.50,
        0.56,0.67,0.58,0.44,0.55,
        0.79,0.57,0.67,0.62,0.76
    ],
    'acousticness': [
        0.01,0.07,0.42,0.30,0.02,
        0.06,0.12,0.08,0.02,0.19,
        0.04,0.25,0.02,0.02,0.01,
        0.00,0.06,0.08,0.72,0.64,
        0.10,0.01,0.13,0.04,0.37
    ]
}

df = pd.DataFrame(songs_data)
print(f'Dataset loaded: {df.shape[0]} songs x {df.shape[1]} features')
print()
print(df[['track_name','artist','genre','bpm','energy','valence','danceability']].to_string(index=False))

## Step 4: Collaborative Filtering — Cosine Similarity

In [ ]:
# Normalize audio features for similarity calculation
feature_cols = ['energy', 'valence', 'danceability', 'acousticness']

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(df_scaled[feature_cols])
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df['track_name'],
    columns=df['track_name']
)

print('Cosine similarity matrix computed!')
print(f'Matrix shape: {similarity_df.shape}')
print()
print('Sample similarities for Blinding Lights:')
print(similarity_df['Blinding Lights'].sort_values(ascending=False).head(6))

## Step 5: Playlist Generator Function

In [ ]:
def generate_playlist(seed_song, mood=None, genre=None, n=8):
    """
    Generate a personalized playlist using collaborative filtering.
    
    Parameters:
        seed_song (str): Song name to base recommendations on
        mood      (str): Optional mood filter (Happy/Chill/Energetic/Sad)
        genre     (str): Optional genre filter
        n         (int): Number of songs to return
    
    Returns:
        DataFrame: Recommended playlist
    """
    if seed_song not in similarity_df.columns:
        print(f'Song "{seed_song}" not found. Available: {list(df["track_name"][:5])}...')
        return None

    # Get similarity scores for seed song
    scores = similarity_df[seed_song].drop(seed_song).sort_values(ascending=False)
    recommended = df[df['track_name'].isin(scores.index)].copy()
    recommended['similarity_score'] = recommended['track_name'].map(scores)

    # Apply mood filter
    mood_filters = {
        'Happy':     recommended['valence'] > 0.6,
        'Chill':     recommended['energy']  < 0.5,
        'Energetic': recommended['energy']  > 0.7,
        'Sad':       recommended['valence'] < 0.4,
    }
    if mood and mood in mood_filters:
        recommended = recommended[mood_filters[mood]]

    # Apply genre filter
    if genre:
        recommended = recommended[recommended['genre'] == genre]

    result = recommended.sort_values('similarity_score', ascending=False).head(n)
    result['similarity_pct'] = (result['similarity_score'] * 100).round(1).astype(str) + '%'
    return result[['track_name','artist','genre','bpm','energy','valence','danceability','similarity_pct']]


# Test the function
print('===== GENERATED PLAYLIST =====')
print('Seed song: Blinding Lights | Mood: Happy | Genre: Pop')
print()
playlist = generate_playlist('Blinding Lights', mood='Happy', genre='Pop', n=8)
if playlist is not None:
    print(playlist.to_string(index=False))

## Step 6: Visualizations

In [ ]:
# --- Plot 1: Similarity Heatmap ---
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(14, 10))

# Show top 12 songs for clarity
top_songs = df['track_name'].head(12).tolist()
heat_data = similarity_df.loc[top_songs, top_songs]

sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax, vmin=0, vmax=1,
            annot_kws={'size': 8})

ax.set_title('Collaborative Filtering — Song Similarity Heatmap', fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('plot1_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved.')

In [ ]:
# --- Plot 2: Audio Feature Distribution by Genre ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
features = ['energy', 'valence', 'danceability']
titles   = ['Energy by Genre', 'Valence (Mood) by Genre', 'Danceability by Genre']
colors   = ['#D85A30', '#185FA5', '#1D9E75']

for i, (feat, title, color) in enumerate(zip(features, titles, colors)):
    df.groupby('genre')[feat].mean().sort_values().plot(
        kind='barh', ax=axes[i], color=color, alpha=0.85
    )
    axes[i].set_title(title, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Average Score', fontsize=9)
    axes[i].set_xlim(0, 1)
    axes[i].tick_params(labelsize=9)

plt.suptitle('Audio Feature Comparison by Genre', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot2_features_by_genre.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved.')

In [ ]:
# --- Plot 3: Energy vs Valence Scatter (mood map) ---
fig, ax = plt.subplots(figsize=(10, 7))

genre_colors = {'Pop':'#D85A30','Hip-Hop':'#185FA5','Rock':'#1D9E75','Electronic':'#BA7517','Jazz':'#7F77DD'}
for genre, group in df.groupby('genre'):
    ax.scatter(group['valence'], group['energy'],
               c=genre_colors.get(genre, '#888'),
               label=genre, s=100, alpha=0.85, edgecolors='white', linewidth=0.5)
    for _, row in group.iterrows():
        ax.annotate(row['track_name'], (row['valence'], row['energy']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points', alpha=0.75)

ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(0.08, 0.92, 'Angry / Intense', transform=ax.transAxes, fontsize=9, color='gray')
ax.text(0.62, 0.92, 'Happy / Excited',  transform=ax.transAxes, fontsize=9, color='gray')
ax.text(0.08, 0.05, 'Sad / Depressed',  transform=ax.transAxes, fontsize=9, color='gray')
ax.text(0.62, 0.05, 'Calm / Relaxed',   transform=ax.transAxes, fontsize=9, color='gray')

ax.set_xlabel('Valence (Positivity)', fontsize=11)
ax.set_ylabel('Energy', fontsize=11)
ax.set_title('Mood Map — Energy vs Valence', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='center right', fontsize=9)
plt.tight_layout()
plt.savefig('plot3_mood_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved.')

In [ ]:
# --- Plot 4: Generated Playlist — Similarity Scores Bar Chart ---
playlist = generate_playlist('Blinding Lights', mood='Happy', n=8)

fig, ax = plt.subplots(figsize=(10, 5))
scores = [float(p.replace('%','')) for p in playlist['similarity_pct']]
names  = [f"{r['track_name']}\n{r['artist']}" for _, r in playlist.iterrows()]
colors = ['#185FA5' if s > 90 else '#1D9E75' if s > 80 else '#BA7517' for s in scores]

bars = ax.barh(names, scores, color=colors, alpha=0.88, edgecolor='white')
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() - 2, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}%', va='center', ha='right', fontsize=9,
            color='white', fontweight='bold')

ax.set_xlabel('Similarity Score (%)', fontsize=11)
ax.set_title('Recommended Playlist — Similarity Scores\nSeed: Blinding Lights | Mood: Happy',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlim(0, 105)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plot4_playlist_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved.')

In [ ]:
# --- Plot 5: Correlation Heatmap of Audio Features ---
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[['bpm','energy','valence','danceability','acousticness']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Audio Feature Correlation Matrix', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('plot5_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 5 saved.')

## Step 7: Final Summary

In [ ]:
print('================================================')
print('    MUSIC PLAYLIST GENERATOR — SUMMARY         ')
print('================================================')
print(f'Total songs in dataset     : {len(df)}')
print(f'Genres covered             : {", ".join(df["genre"].unique())}')
print(f'Feature dimensions         : {len(feature_cols)} (energy, valence, danceability, acousticness)')
print(f'Algorithm used             : Cosine Similarity (Collaborative Filtering)')
print()
print('Sample playlist (seed: Blinding Lights | mood: Happy):')
pl = generate_playlist('Blinding Lights', mood='Happy', n=5)
for _, row in pl.iterrows():
    print(f"  {row['track_name']:<28} {row['artist']:<20} Match: {row['similarity_pct']}")
print()
print('Key Insights:')
print('  - Cosine similarity captures audio feature proximity accurately')
print('  - Mood filtering narrows results to emotionally relevant songs')
print('  - Pop and Electronic genres share highest energy-danceability correlation')
print('  - Valence and acousticness are weakly correlated (r = -0.12)')
print('================================================')